#### Simple Gen AI APP Using Langchain

In [26]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
## Langsmith Tracking
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [27]:
## Data Ingestion--From the website we need to scrape the data
from langchain_community.document_loaders import WebBaseLoader

In [28]:
web_url = "https://www.newindianexpress.com/india/2026/Aug/15/jharkhand-protest-students-to-burn-soren-and-rahul-effigies-gherao-cms-residence-august-20"
loader=WebBaseLoader(web_url)
loader

In [29]:
docs=loader.load()
docs

[Document(metadata={'source': 'https://www.newindianexpress.com/india/2026/Aug/15/jharkhand-protest-students-to-burn-soren-and-rahul-effigies-gherao-cms-residence-august-20', 'title': "Jharkhand protest: Students to burn Soren and Rahul effigies, gherao CM's residence August 20", 'description': 'Protesters said they will demand Soren\'s resignation and urged Rahul Gandhi to withdraw support from the JMM-led coalition if the "CM is not listening" to him.', 'language': 'en'}, page_content='\n\n\n\nJharkhand protest: Students to burn Soren and Rahul effigies, gherao CM\'s residence August 20\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSubscribeE-PAPERINDIAWORLDSTATESCITIESOPINIONEXPLAINERBUSINESSSPORTSGOOD NEWSENTERTAINMENTVIDEOSLIVE BLOGINDIAWORLDSTATESCITIESOPINIONEXPLAINERBUSINESSSPORTSGOOD NEWSENTERTAINMENTVIDEOSLIVE BLOG\n\n\n\n\n\n\nIndiaJharkhand protest: Students to burn Soren and Rahul effigies, gherao CM\'s residence August 

In [30]:
### Load Data--> Docs-->Divide our Docuemnts into chunks dcouments-->text-->vectors-->Vector Embeddings--->Vector Store DB
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
documents=text_splitter.split_documents(docs)

In [31]:
documents

[Document(metadata={'source': 'https://www.newindianexpress.com/india/2026/Aug/15/jharkhand-protest-students-to-burn-soren-and-rahul-effigies-gherao-cms-residence-august-20', 'title': "Jharkhand protest: Students to burn Soren and Rahul effigies, gherao CM's residence August 20", 'description': 'Protesters said they will demand Soren\'s resignation and urged Rahul Gandhi to withdraw support from the JMM-led coalition if the "CM is not listening" to him.', 'language': 'en'}, page_content="Jharkhand protest: Students to burn Soren and Rahul effigies, gherao CM's residence August 20\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSubscribeE-PAPERINDIAWORLDSTATESCITIESOPINIONEXPLAINERBUSINESSSPORTSGOOD NEWSENTERTAINMENTVIDEOSLIVE BLOGINDIAWORLDSTATESCITIESOPINIONEXPLAINERBUSINESSSPORTSGOOD NEWSENTERTAINMENTVIDEOSLIVE BLOG"),
 Document(metadata={'source': 'https://www.newindianexpress.com/india/2026/Aug/15/jharkhand-protest-students-to-burn

In [32]:
from langchain_ollama import OllamaEmbeddings
embeddings=OllamaEmbeddings(model="embeddinggemma")

In [33]:
from langchain_community.vectorstores import FAISS
vectorstoredb=FAISS.from_documents(documents,embeddings)

In [34]:
vectorstoredb

In [35]:
## Query From a vector db
query="LangSmith has two usage limits: total traces and extended"
result=vectorstoredb.similarity_search(query)
result[0].page_content

'Copyright - newindianexpress.com 2026. All rights reserved. Powered by Quintype\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nX\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nThe New Indian Express\n www.newindianexpress.com \nINSTALL APP'

In [36]:
from langchain_ollama import ChatOllama
model = ChatOllama(model="gemma:2b")
model

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.5.5', 'langchain': '1.3.15'}}, model='gemma:2b')

In [37]:
## Retrieval Chain, Document chain

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_template(
    """
Answer the following question based only on the provided context:
<context>
{context}
</context>


"""
)

document_chain=create_stuff_documents_chain(model,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n\n\n'), additional_kwargs={})])
| ChatOllama(metadata={'lc_versions': {'langchain-core': '1.5.5', 'langchain': '1.3.15'}}, model='gemma:2b')
| StrOutputParser(), kwargs={}, config={'run_name': 'stuff_documents_chain'}, config_factories=[])

In [38]:
from langchain_core.documents import Document
document_chain.invoke({
    "input":"LangSmith has two usage limits: total traces and extended",
    "context":[Document(page_content="LangSmith has two usage limits: total traces and extended traces. These correspond to the two metrics we've been tracking on our usage graph. ")]
})

'Sure, based on the context, the two usage limits are:\n\n* Total traces\n* Extended traces'

However, we want the documents to first come from the retriever we just set up. That way, we can use the retriever to dynamically select the most relevant documents and pass those in for a given question.

In [39]:
### Input--->Retriever--->vectorstoredb

vectorstoredb

In [40]:
retriever=vectorstoredb.as_retriever()
from langchain_classic.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,document_chain)


In [41]:
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002923DD12990>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n\n\n'), additional_kwargs={})])
            | 

In [42]:
## Get the response form the LLM
response=retrieval_chain.invoke({"input":"whose effigies are being burnt in jharkhand protest?"})
response['answer']

"Sure, here's the answer to the question:\n\nThe students plan to burn effigies of the chief minister and Congress leader Rahul Gandhi across all 24 districts of the state on Sunday, August 20, demanding his resignation."

In [43]:

response

{'input': 'whose effigies are being burnt in jharkhand protest?',
 'context': [Document(id='66d5b640-a789-4a13-a6ab-f207db100197', metadata={'source': 'https://www.newindianexpress.com/india/2026/Aug/15/jharkhand-protest-students-to-burn-soren-and-rahul-effigies-gherao-cms-residence-august-20', 'title': "Jharkhand protest: Students to burn Soren and Rahul effigies, gherao CM's residence August 20", 'description': 'Protesters said they will demand Soren\'s resignation and urged Rahul Gandhi to withdraw support from the JMM-led coalition if the "CM is not listening" to him.', 'language': 'en'}, page_content='IndiaJharkhand protest: Students to burn Soren and Rahul effigies, gherao CM\'s residence August 20Protesters said they will demand Soren\'s resignation and urged Rahul Gandhi to withdraw support from the JMM-led coalition if the "CM is not listening" to him.Job aspirants and students taking out a massive \'Tiranga Yatra\' procession to protest against the Jharkhand government over t

In [44]:
response['context']

[Document(id='66d5b640-a789-4a13-a6ab-f207db100197', metadata={'source': 'https://www.newindianexpress.com/india/2026/Aug/15/jharkhand-protest-students-to-burn-soren-and-rahul-effigies-gherao-cms-residence-august-20', 'title': "Jharkhand protest: Students to burn Soren and Rahul effigies, gherao CM's residence August 20", 'description': 'Protesters said they will demand Soren\'s resignation and urged Rahul Gandhi to withdraw support from the JMM-led coalition if the "CM is not listening" to him.', 'language': 'en'}, page_content='IndiaJharkhand protest: Students to burn Soren and Rahul effigies, gherao CM\'s residence August 20Protesters said they will demand Soren\'s resignation and urged Rahul Gandhi to withdraw support from the JMM-led coalition if the "CM is not listening" to him.Job aspirants and students taking out a massive \'Tiranga Yatra\' procession to protest against the Jharkhand government over the JPSC and JSSC-CGL exam paper leak issues, in Ranchi, Saturday, Aug. 15, 202

In [47]:
response=retrieval_chain.invoke({"input":"what is the demand of students?"})
print(response['answer'])

Sure, here is the answer to the question:

The students' platform has sought a CBI investigation into the alleged exam irregularities in Jharkhand's recruitment examinations.
